# Model Health Check

Quick diagnostic checks: weight norms, activation ranges, attention health,
prediction quality, and residual growth.

## Why This Matters

Model health check provides tools for systematic analysis and comparison of transformer internals. These diagnostics help identify unexpected behaviors, compare architectures, and build a comprehensive picture of how the model processes information.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.model_health_check import (
    weight_norm_check, activation_range_check,
    attention_health_check, prediction_quality_check,
    residual_growth_check,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Weight Norms

Are weight magnitudes reasonable and consistent across layers?

In [ ]:
result = weight_norm_check(model)
print(f"Embed: {result['embed_norm']:.4f}, Unembed: {result['unembed_norm']:.4f}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: Q={p['W_Q_norm']:.3f} K={p['W_K_norm']:.3f} "
          f"V={p['W_V_norm']:.3f} O={p['W_O_norm']:.3f} "
          f"In={p['W_in_norm']:.3f} Out={p['W_out_norm']:.3f}")

## Activation Ranges

Are activations in a reasonable range? Any overflow or dead regions?

In [ ]:
result = activation_range_check(model, tokens)
print(f"Layers with large activations: {result['n_layers_with_large_acts']}\n")
for p in result['per_layer']:
    warn = ' [LARGE]' if p['has_large_activations'] else ''
    print(f"  Layer {p['layer']}: resid_max={p['resid_max']:.4f}, "
          f"mlp_max={p['mlp_max']:.4f}, sparsity={p['mlp_sparsity']:.1%}{warn}")

## Attention Health

Are any attention heads degenerate (collapsed to single token)?

In [ ]:
result = attention_health_check(model, tokens)
print(f"Degenerate heads: {result['n_degenerate']}/{len(result['per_head'])}\n")
for h in result['per_head']:
    deg = ' [DEGENERATE]' if h['is_degenerate'] else ''
    print(f"  L{h['layer']}H{h['head']}: entropy={h['last_pos_entropy']:.4f}, "
          f"max_weight={h['last_pos_max_weight']:.4f}{deg}")

## Prediction Quality

How well does the model predict the next token at each position?

In [ ]:
result = prediction_quality_check(model, tokens)
print(f"Accuracy: {result['accuracy']:.0%} ({result['n_correct']}/{len(result['per_position'])})\n")
for p in result['per_position']:
    correct = '✓' if p['correct'] else '✗'
    print(f"  Pos {p['position']}: top={p['top_token']}({p['top_prob']:.3f}), "
          f"next={p['next_token_rank']}th({p['next_token_prob']:.3f}), "
          f"H={p['entropy']:.3f} [{correct}]")

## Residual Growth

Is the residual stream growing or collapsing through layers?

In [ ]:
result = residual_growth_check(model, tokens)
print(f"Embed norm: {result['embed_norm']:.4f} → Final: {result['final_norm']:.4f}\n")
for p in result['per_layer']:
    warn = ''
    if p['is_exploding']: warn = ' [EXPLODING]'
    if p['is_collapsing']: warn = ' [COLLAPSING]'
    bar = '█' * int(min(p['growth_factor'], 3) * 5)
    print(f"  Layer {p['layer']}: norm={p['mean_norm']:.4f}, "
          f"growth={p['growth_factor']:.2f}x{warn} {bar}")